# Production AICC

Replay transfers, collect keypad input, and size a contact center. Predict each result first.

In [ ]:
import sys
sys.path.insert(0, '../code')
import main

events = [(5, 'transfer_requested'), (6, 'dial_started'), (14, 'human_answered'), (35, 'human_accepted'), (37, 'ai_left')]
trace = main.run_transfer(events)
print(' -> '.join(trace.states), '| hold', trace.hold_seconds(), 's')

late = [(5, 'transfer_requested'), (6, 'dial_started'), (40, 'human_answered')]
print(main.run_transfer(late, ringing_timeout=20).states)

## Keypad input and masking

What happens if the caller pauses 5 s after the third digit?

In [ ]:
presses = [(1.0, '1'), (1.6, '2'), (2.1, '3'), (7.1, '4'), (7.5, '5'), (8.0, '6')]
print(main.collect_digits(presses, num_digits=6))
print(main.mask_pii('Mã đơn 4721, SĐT 0901.234.567, thẻ 4111 1111 1111 1111'))

## KPIs and staffing

Change the peak volume, the AHT or the transfer rate. How many humans does each point of containment save?

In [ ]:
kpis = main.compute_kpis(main.generate_call_log(2000, seed=0))
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in kpis.items()})

for transfer_rate in (0.30, 0.25, 0.20, 0.15):
    transfers = 1200 * transfer_rate
    print(f"transfer rate {transfer_rate:.0%}: {main.agents_for_service_level(transfers, 300)} humans at peak")

In [ ]:
traffic = main.offered_load(1200, 160)
for blocking in (0.05, 0.01, 0.001):
    print(f"{blocking:.1%} blocking -> {main.channels_for_blocking(traffic, blocking)} concurrent sessions")